# RefSeq viral detection

This report is independent of `07_Normalise.py.ipynb`. It uses CoverM's original per-reference mapping statistics and base-level coverage; the existing normalization matrices are retained for cross-checking and RPKM. A RefSeq hit is evidence of similarity to a reference, not proof of species identity. All samples, including an optional negative control and zero-hit samples, remain in the exported tables.

In [ ]:
from pathlib import Path
import json
import os
import re
import sqlite3
import time
from xml.etree import ElementTree

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns

samples = list(snakemake.params.samples)
negative_control = str(snakemake.params.negative_control).strip()
threshold = {
    "high_min_length_bp": int(snakemake.params.high_min_length_bp),
    "high_min_covered_bases": int(snakemake.params.high_min_covered_bases),
    "high_min_breadth_percent": float(snakemake.params.high_min_breadth_percent),
    "high_min_unique_reads": int(snakemake.params.high_min_unique_reads),
    "candidate_min_unique_reads": int(snakemake.params.candidate_min_unique_reads),
    "candidate_min_breadth_percent": float(snakemake.params.candidate_min_breadth_percent),
    "ambiguous_min_reads": int(snakemake.params.ambiguous_min_reads),
    "ambiguous_max_unique_ratio": float(snakemake.params.ambiguous_max_unique_ratio),
    "negative_control_min_reads": int(snakemake.params.negative_control_min_reads),
    "negative_control_max_enrichment": float(snakemake.params.negative_control_max_enrichment),
    "enrichment_pseudocount": float(snakemake.params.enrichment_pseudocount),
}
plot_max_accessions = int(snakemake.params.plot_max_accessions)
figures_dir = Path(snakemake.output.figures)
figures_dir.mkdir(parents=True, exist_ok=True)
for old_name in ["reported_rpkm_heatmap", "rna_dna_composition", "unique_mapping_ratio"]:
    for suffix in ["png", "svg"]:
        (figures_dir / f"{old_name}.{suffix}").unlink(missing_ok=True)
for path in [snakemake.output.long, snakemake.output.matrix, snakemake.output.metadata, snakemake.output.high, snakemake.output.candidates, snakemake.output.contaminants]:
    Path(path).parent.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="ticks", palette="colorblind", context="paper")
plt.rcParams.update({"savefig.dpi": 300, "svg.fonttype": "none", "font.family": "sans-serif"})
class_order = ["high_confidence", "candidate", "ambiguous", "not_detected"]
class_colors = {"high_confidence": "#228833", "candidate": "#4477AA", "ambiguous": "#CC6677", "not_detected": "#BBBBBB"}
print("Samples:", len(samples), "| Negative control:", negative_control or "not configured")
print("Detection thresholds:", threshold)

## CoverM mapping statistics and base-level coverage

CoverM's `Read Count` and `Mean` are used directly. The workflow's RefSeq mean-depth matrix also retains CoverM's `Mean` values; it is checked against the per-sample files. Base-coverage files contain only positions with positive depth; positional span helps distinguish distributed candidate coverage from a small hotspot.

In [ ]:
metrics = ["Mean", "Length", "Covered Bases", "Read Count", "RPKM"]

def read_coverm(path, sample, unique=False):
    table = pd.read_csv(path, sep="\t", dtype={"Contig": str})
    columns = {}
    for metric in metrics:
        matches = [column for column in table if column.endswith(" " + metric) and not (metric == "Mean" and column.endswith(" Trimmed Mean"))]
        columns[matches[0]] = ("unique_" if unique else "") + {
            "Mean": "mean_depth", "Length": "genome_length",
            "Covered Bases": "covered_bases", "Read Count": "mapped_read_count",
            "RPKM": "coverm_rpkm",
        }[metric]
    table = table[["Contig"] + list(columns)].rename(columns={"Contig": "refseq_accession", **columns})
    for column in columns.values():
        table[column] = pd.to_numeric(table[column], errors="coerce").fillna(0)
    table["sample"] = sample
    return table

def base_coverage_summary(path):
    summary = {}
    with open(path) as handle:
        for line in handle:
            fields = line.rstrip("\n").split("\t")
            accession, position, depth = fields
            position = int(position)
            current = summary.get(accession)
            if current is None:
                summary[accession] = [1, position, position]
            else:
                current[0] += 1
                current[1] = min(current[1], position)
                current[2] = max(current[2], position)
    return summary

sample_tables = []
for sample, all_path, unique_path, base_path, unique_base_path in zip(
    samples, snakemake.input.covstats, snakemake.input.unique_covstats,
    snakemake.input.basecov, snakemake.input.unique_basecov,
):
    all_stats = read_coverm(all_path, sample)
    unique_stats = read_coverm(unique_path, sample, unique=True)
    table = all_stats.merge(unique_stats.drop(columns="sample"), on="refseq_accession")
    table = table.drop(columns=["unique_genome_length"])
    base = base_coverage_summary(base_path)
    unique_base = base_coverage_summary(unique_base_path)
    table["basecov_covered_bases"] = table["refseq_accession"].map(lambda accession: base.get(accession, [0, 0, 0])[0])
    table["unique_basecov_covered_bases"] = table["refseq_accession"].map(lambda accession: unique_base.get(accession, [0, 0, 0])[0])
    table["coverage_span_percent"] = table.apply(
        lambda row: 100 * (base[row["refseq_accession"]][2] - base[row["refseq_accession"]][1] + 1) / row["genome_length"]
        if row["refseq_accession"] in base else 0.0, axis=1
    )
    sample_tables.append(table)
    print(sample, "mapped accessions:", int((table["mapped_read_count"] > 0).sum()))
long = pd.concat(sample_tables, ignore_index=True)
reference_ids = set(long["refseq_accession"])
long["breadth_percent"] = 100 * long["covered_bases"] / long["genome_length"]
long["unique_breadth_percent"] = 100 * long["unique_covered_bases"] / long["genome_length"]
long["unique_all_read_ratio"] = np.divide(
    long["unique_mapped_read_count"], long["mapped_read_count"],
    out=np.zeros(len(long), dtype=float), where=long["mapped_read_count"].to_numpy() > 0
)

## Existing RefSeq matrices

These matrices provide the workflow RPKM and supporting abundance measurements.


In [ ]:
def read_workflow_matrix(path, value_name):
    frame = pd.read_csv(path, index_col=0)
    frame.index = frame.index.astype(str)
    frame = frame.loc[:, samples].apply(pd.to_numeric, errors="coerce").fillna(0)
    frame.index.name = "refseq_accession"
    return frame.reset_index().melt(id_vars="refseq_accession", var_name="sample", value_name=value_name)

for path, name in [
    (snakemake.input.rpkm, "rpkm"),
    (snakemake.input.counts, "workflow_mapped_read_count"),
    (snakemake.input.breadth, "workflow_breadth_percent"),
    (snakemake.input.depth, "workflow_mean_depth"),
]:
    long = long.merge(read_workflow_matrix(path, name), on=["refseq_accession", "sample"])


## RefSeq metadata

The notebook downloads metadata in batches from NCBI Virus, Nucleotide and Taxonomy using Python and caches responses in SQLite. Genome types use NCBI Virus molecule type when present, then specific ICTV taxa reported in the NCBI lineage, then NCBI Nucleotide molecule/strand fields. Ambiguous DNA/RNA records stay generic rather than being assigned an unsupported strand. ICTV references: [Duplodnaviria](https://ictv.global/report/chapter/duplodnaviria/duplodnaviria), [Circoviridae](https://ictv.global/report/chapter/circoviridae/circoviridae), [Parvoviridae](https://ictv.global/report/chapter/parvoviridae/parvoviridae), [Microviridae](https://ictv.global/report_9th/ssDNA/Microviridae), and [Polycipiviridae](https://ictv.global/report/chapter/polycipiviridae/polycipiviridae).


In [ ]:
metadata_columns = ["refseq_accession", "virus_name", "taxid", "realm", "family", "genus", "species",
                    "genome_length", "genome_type", "genome_type_source", "host", "host_taxid", "refseq_description", "ncbi_status"]
fasta_descriptions = {}
with open(snakemake.input.fasta) as handle:
    for line in handle:
        if line.startswith(">"):
            identifier, _, description = line[1:].strip().partition(" ")
            if identifier in reference_ids:
                fasta_descriptions[identifier] = description
lengths = long[["refseq_accession", "genome_length"]].drop_duplicates()
# NCBI Virus, Nucleotide and Taxonomy are queried in accession/taxid batches.
# The internal SQLite cache is resumable and never replaces the local FASTA.
cache_path = Path(snakemake.params.metadata_cache)
cache_path.parent.mkdir(parents=True, exist_ok=True)
database = sqlite3.connect(cache_path)
for table in ["virus", "nucleotide", "taxonomy"]:
    database.execute(f"CREATE TABLE IF NOT EXISTS {table} (key TEXT PRIMARY KEY, payload TEXT NOT NULL)")
if bool(snakemake.params.metadata_refresh):
    for table in ["virus", "nucleotide", "taxonomy"]:
        database.execute(f"DELETE FROM {table}")
    database.commit()

def cached(table):
    return {key: json.loads(payload) for key, payload in database.execute(f"SELECT key, payload FROM {table}")}

def save_batch(table, values):
    database.executemany(
        f"INSERT OR REPLACE INTO {table} (key, payload) VALUES (?, ?)",
        [(key, json.dumps(value, separators=(",", ":"))) for key, value in values.items()]
    )
    database.commit()

def batches(values, size=200):
    values = sorted(values)
    for start in range(0, len(values), size):
        yield values[start:start + size]

api_key = os.environ.get("NCBI_API_KEY", "").strip()
email = os.environ.get("NCBI_EMAIL", "").strip()
session = requests.Session()
session.headers.update({"User-Agent": "MOSAIC-RefSeq-detection/1.0"})
if api_key:
    session.headers["api-key"] = api_key

def ncbi_request(method, url, *, params=None, data=None, missing_ok=False):
    for attempt in range(5):
        time.sleep(0.36)
        try:
            response = session.request(method, url, params=params, data=data, timeout=(15, 90))
            if response.status_code == 200:
                return response
            if response.status_code == 404 and missing_ok:
                return None
        except requests.RequestException:
            pass
        time.sleep(min(30, 2 ** attempt))
    print("NCBI metadata unavailable:", url)
    return None

accessions = sorted(reference_ids)
virus_cache = cached("virus")
missing = set(accessions) - set(virus_cache)
print("NCBI Virus metadata: cached", len(reference_ids) - len(missing), "to fetch", len(missing))
for batch in batches(missing):
    url = "https://api.ncbi.nlm.nih.gov/datasets/v2/virus/accession/" + ",".join(batch) + "/dataset_report"
    response = ncbi_request("GET", url, params={"page_size": 1000}, missing_ok=True)
    payload = response.json() if response is not None else {}
    if response is not None:
        reports = {record["accession"]: record for record in payload.get("reports", [])}
        save_batch("virus", {accession: reports.get(accession) for accession in batch})
virus_cache = cached("virus")

nucleotide_cache = cached("nucleotide")
missing = set(accessions) - set(nucleotide_cache)
print("NCBI Nucleotide metadata: cached", len(reference_ids) - len(missing), "to fetch", len(missing))
eutils = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
for batch in batches(missing):
    data = {"db": "nuccore", "id": ",".join(batch), "retmode": "json", "tool": "MOSAIC"}
    if email:
        data["email"] = email
    if api_key:
        data["api_key"] = api_key
    response = ncbi_request("POST", eutils + "esummary.fcgi", data=data)
    if response is None:
        continue
    payload = response.json()
    result = payload.get("result", {})
    documents = {}
    for uid in result.get("uids", []):
        record = result.get(str(uid), {})
        accession = record.get("accessionversion")
        if accession in batch:
            documents[accession] = {
                name: record.get(name) for name in
                ["taxid", "moltype", "strand", "organism", "title", "slen"]
            }
    save_batch("nucleotide", {accession: documents.get(accession) for accession in batch})
nucleotide_cache = cached("nucleotide")

taxids = set()
for accession in accessions:
    report = virus_cache.get(accession) or {}
    summary = nucleotide_cache.get(accession) or {}
    taxid = (report.get("virus") or {}).get("tax_id") or summary.get("taxid")
    if taxid:
        taxids.add(str(taxid))
taxonomy_cache = cached("taxonomy")
missing = taxids - set(taxonomy_cache)
print("NCBI Taxonomy records: cached", len(taxids) - len(missing), "to fetch", len(missing))
for batch in batches(missing):
    data = {"db": "taxonomy", "id": ",".join(batch), "retmode": "xml", "tool": "MOSAIC"}
    if email:
        data["email"] = email
    if api_key:
        data["api_key"] = api_key
    response = ncbi_request("POST", eutils + "efetch.fcgi", data=data)
    if response is None:
        continue
    root = ElementTree.fromstring(response.content)
    records = {}
    for taxon in root.findall("./Taxon"):
        taxid = taxon.findtext("TaxId")
        ranks = {}
        for ancestor in list(taxon.findall("./LineageEx/Taxon")) + [taxon]:
            rank = ancestor.findtext("Rank")
            if rank in ["realm", "family", "genus", "species"]:
                ranks[rank] = ancestor.findtext("ScientificName")
        if taxid in batch:
            records[taxid] = ranks
    save_batch("taxonomy", {taxid: records.get(taxid) for taxid in batch})
taxonomy_cache = cached("taxonomy")
database.close()
session.close()

rows = []
for accession in accessions:
    description = fasta_descriptions.get(accession, "")
    fallback_name = re.sub(r",?\s+(?:complete genome|complete sequence).*$", "", description, flags=re.IGNORECASE)
    report = virus_cache.get(accession) or {}
    summary = nucleotide_cache.get(accession) or {}
    virus = report.get("virus") or {}
    host = report.get("host") or {}
    taxid = virus.get("tax_id") or summary.get("taxid")
    ranks = taxonomy_cache.get(str(taxid)) or {}
    lineage = {item.get("name") for item in virus.get("lineage", [])}
    molecule = str(summary.get("moltype") or "").upper()
    strand = str(summary.get("strand") or "").lower()
    strand = {"single": "ss", "double": "ds"}.get(strand, strand)
    reported_type = str(report.get("mol_type") or "").strip()
    if reported_type:
        genome_type, genome_type_source = reported_type, "NCBI Virus"
    elif "Duplodnaviria" in lineage:
        genome_type, genome_type_source = "dsDNA", "ICTV taxonomy"
    elif lineage.intersection({"Circoviridae", "Parvoviridae", "Phixviricota"}):
        genome_type, genome_type_source = "ssDNA", "ICTV taxonomy"
    elif "Polycipiviridae" in lineage:
        genome_type, genome_type_source = "ssRNA(+)", "ICTV taxonomy"
    else:
        genome_type = strand + molecule if molecule in ["RNA", "DNA"] and strand in ["ss", "ds"] else molecule
        genome_type_source = "NCBI Nucleotide" if genome_type else None
    rows.append({
        "refseq_accession": accession,
        "virus_name": virus.get("organism_name") or summary.get("organism") or fallback_name,
        "taxid": str(taxid) if taxid else None,
        "realm": ranks.get("realm"), "family": ranks.get("family"),
        "genus": ranks.get("genus"), "species": ranks.get("species"),
        "genome_type": genome_type or None,
        "genome_type_source": genome_type_source,
        "host": host.get("organism_name"), "host_taxid": host.get("tax_id"),
        "refseq_description": description,
        "ncbi_status": "found" if report or summary else "not_found",
    })
metadata = pd.DataFrame(rows).merge(lengths, on="refseq_accession")
metadata = metadata[metadata_columns]
metadata.to_csv(snakemake.output.metadata, sep="\t", index=False, na_rep="")
long = long.merge(metadata, on=["refseq_accession", "genome_length"])
print("NCBI metadata found for", int(metadata["ncbi_status"].eq("found").sum()),
      "of", len(metadata), "RefSeq accessions; cache:", cache_path)

## Detection classes and negative-control comparison

High confidence uses the existing MOSAIC length/breadth filter and at least five unique mapped reads. Candidates require at least ten unique reads and 25% genome breadth. The negative control is assessed with these same evidence thresholds: accessions that qualify there are flagged as contaminants and listed separately. For other samples, a signal comparable to the negative control cannot be high-confidence or candidate. Remaining signals are ambiguous only when at least five reads map and uniqueness is low, the control signal is comparable, or the row is a lower-support negative-control signal. Weaker signals are not detected. The RPKM pseudocount keeps control enrichment finite when the control has zero RPKM.


In [ ]:
long["is_negative_control"] = long["sample"].eq(negative_control) if negative_control else False
if negative_control:
    control = long.loc[long["sample"].eq(negative_control), [
        "refseq_accession", "mapped_read_count", "breadth_percent", "rpkm"
    ]].rename(columns={
        "mapped_read_count": "negative_control_reads",
        "breadth_percent": "negative_control_breadth_percent",
        "rpkm": "negative_control_rpkm",
    })
    long = long.merge(control, on="refseq_accession")
    pseudocount = threshold["enrichment_pseudocount"]
    long["sample_negative_control_enrichment"] = (
        (long["rpkm"] + pseudocount) / (long["negative_control_rpkm"] + pseudocount)
    )
    long["negative_control_comparable"] = (
        long["negative_control_reads"].ge(threshold["negative_control_min_reads"])
        & long["sample_negative_control_enrichment"].le(threshold["negative_control_max_enrichment"])
    )
else:
    for name in ["negative_control_reads", "negative_control_breadth_percent",
                 "negative_control_rpkm", "sample_negative_control_enrichment"]:
        long[name] = np.nan
    long["negative_control_comparable"] = False

strict_coverage = (
    (long["genome_length"].ge(threshold["high_min_length_bp"])
     & long["covered_bases"].gt(threshold["high_min_covered_bases"]))
    | (long["genome_length"].lt(threshold["high_min_length_bp"])
       & long["breadth_percent"].gt(threshold["high_min_breadth_percent"]))
)
high = (
    strict_coverage
    & long["unique_mapped_read_count"].ge(threshold["high_min_unique_reads"])
    & (~long["negative_control_comparable"] | long["is_negative_control"])
)
candidate_support = (
    long["unique_mapped_read_count"].ge(threshold["candidate_min_unique_reads"])
    & long["breadth_percent"].ge(threshold["candidate_min_breadth_percent"])
)
candidate = candidate_support & ~high & (~long["negative_control_comparable"] | long["is_negative_control"])
low_unique = long["unique_all_read_ratio"].le(threshold["ambiguous_max_unique_ratio"])
ambiguous = (
    long["mapped_read_count"].ge(threshold["ambiguous_min_reads"])
    & (low_unique | long["negative_control_comparable"] | long["is_negative_control"])
    & ~high & ~candidate
)
long["detection_class"] = np.select(
    [high, candidate, ambiguous],
    ["high_confidence", "candidate", "ambiguous"], default="not_detected"
)
long["strict_coverage_pass"] = strict_coverage
long["candidate_coverage_pass"] = candidate_support
contaminant_table = long.loc[
    long["is_negative_control"] & long["detection_class"].isin(["high_confidence", "candidate"])
].copy()
contaminant_ids = set(contaminant_table["refseq_accession"])
long["is_contaminant"] = long["refseq_accession"].isin(contaminant_ids)
contaminant_table["is_contaminant"] = True
metadata["is_contaminant"] = metadata["refseq_accession"].isin(contaminant_ids)
metadata.to_csv(snakemake.output.metadata, sep="\t", index=False, na_rep="")
long = long.sort_values(["refseq_accession", "sample"]).reset_index(drop=True)
matrix = long.pivot(index="refseq_accession", columns="sample", values="detection_class").reindex(columns=samples)
matrix.index.name = "refseq_accession"
long.to_csv(snakemake.output.long, sep="\t", index=False, na_rep="")
matrix.to_csv(snakemake.output.matrix, sep="\t")
long.loc[long["detection_class"].eq("high_confidence")].to_csv(snakemake.output.high, sep="\t", index=False, na_rep="")
long.loc[long["detection_class"].eq("candidate")].to_csv(snakemake.output.candidates, sep="\t", index=False, na_rep="")
contaminant_table.to_csv(snakemake.output.contaminants, sep="\t", index=False, na_rep="")
print(long["detection_class"].value_counts().reindex(class_order, fill_value=0))

## Detection figures

Both heatmaps display all available RPKM or breadth measurements across samples for the selected RefSeq accessions, including values in ambiguous and not-detected cells. Row labels end with the bold reference length in bp. For ordinary rows, two stars mark high-confidence calls and one star marks candidates. In red contaminant rows, stars show which samples meet the same mapping-evidence thresholds before the negative-control comparison; those samples may still be classified as ambiguous in the results table. No star means no reported call for ordinary rows, or neither mapping-evidence threshold is met in a red contaminant row. RPKM is log-transformed; breadth is shown as a percentage. Non-control accessions are selected by maximum reported RPKM up to the configured limit. Accessions classified as high-confidence or candidate in the negative control are additionally shown at the bottom, regardless of that limit; their names and stars are red. Within each group, references run from shortest to longest (accession breaks ties). The contaminant table records the qualifying negative-control rows; a red label means NC evidence, not proof of a laboratory contamination source. PNG and SVG figures are also saved.


In [ ]:
evidence = long.loc[long["mapped_read_count"].gt(0)].copy()
reported = evidence.loc[evidence["detection_class"].isin(["high_confidence", "candidate"])].copy()
if negative_control:
    reported = reported.loc[~reported["is_negative_control"]]
reported = reported.loc[~reported["is_contaminant"]]
normal_ids = (
    reported.groupby("refseq_accession")["rpkm"].max()
    .sort_values(ascending=False).head(plot_max_accessions).index
)
name_map = metadata.set_index("refseq_accession")["virus_name"].fillna("").to_dict()
length_map = metadata.set_index("refseq_accession")["genome_length"].to_dict()
normal_ids = sorted(normal_ids, key=lambda accession: (length_map[accession], accession))
contaminant_ids = sorted(contaminant_ids, key=lambda accession: (length_map[accession], accession))
top_ids = normal_ids + contaminant_ids
label_map = {
    accession: accession + "  " + str(name_map.get(accession, ""))[:55] + "  " + r"$\mathbf{(" + str(int(length_map[accession])) + r"bp)}$"
    for accession in top_ids
}
selected = long.loc[long["refseq_accession"].isin(top_ids)]
selected = selected.copy()
selected["plot_class"] = selected["detection_class"]
contaminant_rows = selected["is_contaminant"]
high_evidence = selected["strict_coverage_pass"] & selected["unique_mapped_read_count"].ge(threshold["high_min_unique_reads"])
selected.loc[contaminant_rows & high_evidence, "plot_class"] = "high_confidence"
selected.loc[contaminant_rows & ~high_evidence & selected["candidate_coverage_pass"], "plot_class"] = "candidate"
classes = selected.pivot(index="refseq_accession", columns="sample", values="plot_class").reindex(index=top_ids, columns=samples)
symbols = {"high_confidence": "**", "candidate": "*"}


### RPKM across samples


In [ ]:
fig, ax = plt.subplots(figsize=(max(8, len(samples) * 0.9 + 2), max(4, len(top_ids) * 0.24 + 2)))
if len(top_ids):
    rpkm = selected.pivot(index="refseq_accession", columns="sample", values="rpkm").reindex(index=top_ids, columns=samples)
    rpkm = np.log10(1 + rpkm)
    rpkm.index = rpkm.index.map(label_map)
    sns.heatmap(rpkm, cmap="mako", ax=ax, vmin=0,
                cbar_kws={"label": "log10(1 + RPKM)"}, linewidths=0.1)
    for row, tick in enumerate(ax.get_yticklabels()):
        if top_ids[row] in contaminant_ids:
            tick.set_color("#C00000")
    for (row, column_index), category in np.ndenumerate(classes.to_numpy()):
        if category in symbols:
            ax.text(column_index + 0.5, row + 0.5, symbols[category], ha="center", va="center",
                    color="#C00000" if top_ids[row] in contaminant_ids else "black",
                    fontsize=12, fontweight="bold",
                    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8, "pad": 0.2})
    ax.set_ylabel("RefSeq accession, virus and genome length")
    ax.set_xlabel("Sample  |  ** high confidence; * candidate; red stars = evidence before NC comparison")
else:
    ax.text(0.5, 0.5, "No high-confidence or candidate detections", ha="center", va="center", transform=ax.transAxes)
    ax.set_axis_off()
ax.set_title("RefSeq viral abundance across all samples")
fig.tight_layout()
fig.savefig(figures_dir / "rpkm_heatmap.png", bbox_inches="tight")
fig.savefig(figures_dir / "rpkm_heatmap.svg", bbox_inches="tight")
plt.show()


### Genome breadth across samples


In [ ]:
fig, ax = plt.subplots(figsize=(max(8, len(samples) * 0.9 + 2), max(4, len(top_ids) * 0.24 + 2)))
if len(top_ids):
    breadth = selected.pivot(index="refseq_accession", columns="sample", values="breadth_percent").reindex(index=top_ids, columns=samples)
    breadth.index = breadth.index.map(label_map)
    sns.heatmap(breadth, cmap="mako", ax=ax, vmin=0, vmax=100,
                cbar_kws={"label": "Genome breadth (%)"}, linewidths=0.1)
    for row, tick in enumerate(ax.get_yticklabels()):
        if top_ids[row] in contaminant_ids:
            tick.set_color("#C00000")
    for (row, column_index), category in np.ndenumerate(classes.to_numpy()):
        if category in symbols:
            ax.text(column_index + 0.5, row + 0.5, symbols[category], ha="center", va="center",
                    color="#C00000" if top_ids[row] in contaminant_ids else "black",
                    fontsize=12, fontweight="bold",
                    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8, "pad": 0.2})
    ax.set_ylabel("RefSeq accession, virus and genome length")
    ax.set_xlabel("Sample  |  ** high confidence; * candidate; red stars = evidence before NC comparison")
else:
    ax.text(0.5, 0.5, "No high-confidence or candidate detections", ha="center", va="center", transform=ax.transAxes)
    ax.set_axis_off()
ax.set_title("RefSeq genome breadth across all samples")
fig.tight_layout()
fig.savefig(figures_dir / "breadth_heatmap.png", bbox_inches="tight")
fig.savefig(figures_dir / "breadth_heatmap.svg", bbox_inches="tight")
plt.show()


### Mapped reads and genome breadth


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
if len(evidence):
    sns.scatterplot(data=evidence, x="mapped_read_count", y="breadth_percent",
                    hue="detection_class", style="detection_class", hue_order=class_order,
                    palette=class_colors, alpha=0.7, s=35, ax=ax)
    ax.set_xscale("log")
    ax.set_ylim(-2, 102)
else:
    ax.text(0.5, 0.5, "No mapped RefSeq viruses", ha="center", va="center", transform=ax.transAxes)
    ax.set_axis_off()
ax.set(xlabel="Mapped read count", ylabel="Genome breadth (%)", title="Read support and genome breadth")
fig.tight_layout()
fig.savefig(figures_dir / "reads_vs_breadth.png", bbox_inches="tight")
fig.savefig(figures_dir / "reads_vs_breadth.svg", bbox_inches="tight")
plt.show()


### Detections per sample


In [ ]:
counts = long.groupby(["sample", "detection_class"]).size().unstack(fill_value=0).reindex(index=samples, columns=class_order, fill_value=0)
fig, ax = plt.subplots(figsize=(max(9, len(samples) * 0.85), 5))
counts[["high_confidence", "candidate", "ambiguous"]].plot.bar(
    stacked=True, color=[class_colors[name] for name in class_order[:3]], ax=ax
)
ax.set(xlabel="Sample", ylabel="RefSeq accessions", title="Viruses by detection class")
ax.legend(title="Detection class")
fig.tight_layout()
fig.savefig(figures_dir / "detections_per_sample.png", bbox_inches="tight")
fig.savefig(figures_dir / "detections_per_sample.svg", bbox_inches="tight")
plt.show()


### Sample versus negative control


In [ ]:
if negative_control:
    comparison = evidence.loc[~evidence["is_negative_control"]]
    fig, ax = plt.subplots(figsize=(9, 6))
    if len(comparison):
        sns.scatterplot(data=comparison, x="negative_control_rpkm", y="rpkm",
                        hue="detection_class", hue_order=class_order, palette=class_colors,
                        alpha=0.7, s=35, ax=ax)
        maximum = max(comparison[["negative_control_rpkm", "rpkm"]].max().max(), 1)
        ax.plot([0, maximum], [0, maximum], "--", color="#555555", linewidth=1, label="1:1")
        ax.set(xscale="symlog", yscale="symlog")
    else:
        ax.text(0.5, 0.5, "No mapped RefSeq viruses outside the control", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
    ax.set(xlabel="Negative-control RPKM", ylabel="Sample RPKM",
           title="Sample versus negative-control abundance")
    fig.tight_layout()
    fig.savefig(figures_dir / "sample_vs_negative_control.png", bbox_inches="tight")
    fig.savefig(figures_dir / "sample_vs_negative_control.svg", bbox_inches="tight")
    plt.show()


### Family composition


In [ ]:
family_frame = reported.copy()
family_frame["family"] = family_frame["family"].fillna("").replace("", "Unassigned")
top_families = family_frame["family"].value_counts().head(12).index.tolist()
family_frame["family"] = family_frame["family"].where(family_frame["family"].isin(top_families), "Other")
family_composition = family_frame.groupby(["sample", "family"]).size().unstack(fill_value=0).reindex(samples, fill_value=0)
fig, ax = plt.subplots(figsize=(max(9, len(samples) * 0.85), 6))
if not family_composition.empty and len(family_composition.columns):
    family_composition.plot.bar(stacked=True, colormap="tab20", ax=ax)
    ax.legend(title="Family", bbox_to_anchor=(1.02, 1), loc="upper left")
else:
    ax.text(0.5, 0.5, "No high-confidence or candidate detections", ha="center", va="center", transform=ax.transAxes)
    ax.set_axis_off()
ax.set(xlabel="Sample", ylabel="Detected RefSeq accessions", title="Detected-virus family composition")
fig.tight_layout()
fig.savefig(figures_dir / "family_composition.png", bbox_inches="tight")
fig.savefig(figures_dir / "family_composition.svg", bbox_inches="tight")
plt.show()


### Genus composition


In [ ]:
genus_frame = reported.copy()
genus_frame["genus"] = genus_frame["genus"].fillna("").replace("", "Unassigned")
top_genera = genus_frame["genus"].value_counts().head(12).index.tolist()
genus_frame["genus"] = genus_frame["genus"].where(genus_frame["genus"].isin(top_genera), "Other")
genus_composition = genus_frame.groupby(["sample", "genus"]).size().unstack(fill_value=0).reindex(samples, fill_value=0)
fig, ax = plt.subplots(figsize=(max(9, len(samples) * 0.85), 6))
if not genus_composition.empty and len(genus_composition.columns):
    genus_composition.plot.bar(stacked=True, colormap="tab20", ax=ax)
    ax.legend(title="Genus", bbox_to_anchor=(1.02, 1), loc="upper left")
else:
    ax.text(0.5, 0.5, "No high-confidence or candidate detections", ha="center", va="center", transform=ax.transAxes)
    ax.set_axis_off()
ax.set(xlabel="Sample", ylabel="Detected RefSeq accessions", title="Detected-virus genus composition")
fig.tight_layout()
fig.savefig(figures_dir / "genus_composition.png", bbox_inches="tight")
fig.savefig(figures_dir / "genus_composition.svg", bbox_inches="tight")
plt.show()


### Genome-type composition


In [ ]:
if len(reported):
    genome_types = reported.copy()
    genome_types["genome_type"] = genome_types["genome_type"].fillna("Unspecified")
    composition = genome_types.groupby(["sample", "genome_type"]).size().unstack(fill_value=0).reindex(samples, fill_value=0)
    type_order = ["ssDNA", "dsDNA", "DNA", "ssRNA(+)", "ssRNA(-)", "ssRNA", "dsRNA", "RNA", "Unspecified"]
    composition = composition.reindex(columns=[name for name in type_order if name in composition] +
                                      [name for name in composition if name not in type_order])
    type_colors = {"ssDNA": "#E69F00", "dsDNA": "#0072B2", "DNA": "#999999",
                   "ssRNA(+)": "#009E73", "ssRNA(-)": "#56B4E9", "ssRNA": "#66CC99",
                   "dsRNA": "#CC79A7", "RNA": "#BBBBBB", "Unspecified": "#DDDDDD"}
    fig, ax = plt.subplots(figsize=(max(9, len(samples) * 0.85), 5))
    composition.plot.bar(stacked=True, color=[type_colors.get(name, "#777777") for name in composition], ax=ax)
    ax.set(xlabel="Sample", ylabel="Detected RefSeq accessions", title="Detected-virus genome types")
    ax.legend(title="Genome type", bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    fig.savefig(figures_dir / "genome_type_composition.png", bbox_inches="tight")
    fig.savefig(figures_dir / "genome_type_composition.svg", bbox_inches="tight")
    plt.show()
else:
    print("Genome-type composition plot skipped: no reported detections.")
print("Tables:", snakemake.output.long, snakemake.output.matrix, snakemake.output.metadata)
print("Figures:", figures_dir)
